# 2025 年组合回测

读取股票预测、行情和指数，模拟每日开盘估值与交易，生成账户、持仓、交易和绩效汇总。运行前先完成上游预测，具体规则见项目根目录 `README.md`。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import cvxpy as cp

## 文件路径与参数

自动向上定位项目根目录，使用项目相对路径读写文件；内核工作目录应位于本项目内。下方同时定义本步骤使用的参数。


In [ ]:
PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'environment.yml').is_file() and (path / 'src').is_dir()
)

STOCK_RETURN = PROJECT_ROOT / "src/stock_return/stock_return.parquet"
PANEL = PROJECT_ROOT / "data/processed/panel.parquet"
RETURN_T2 = PROJECT_ROOT / "src/return_t2/return_t2.parquet"
INDEX = PROJECT_ROOT / "data/processed/index.parquet"


## 读取回测输入

股票预测按前一交易日使用，行情提供开盘价格、停牌和 ST 状态。回测资格不依赖未来收益标签。


In [ ]:
stock_return = pd.read_parquet(STOCK_RETURN)
panel = pd.read_parquet(PANEL)

## 策略与执行口径

按项目根目录 `README.md` 执行。前一日预测、今日开盘状态筛选前 100 名，与合格可交易旧持仓取并集。单股上限为 `1.5 / dim(w)`，其中 `dim(w)` 为去重后的优化候选数量（不含冻结持仓），权重以扣除强制退出费用后的可调配资产为分母。允许现金，固定买卖费率分别为 0.00031、0.00081。持仓使用合成数量，冻结按昨日已知最后有效收盘价估值。不检查无穷值或非停牌价格异常。2025-12-30 开盘仅清仓，冻结持仓保留估值。

## 参数与日期对齐

不依赖收益标签筛选股票。保留全部信号历史，以便回测首日使用前一交易日预测。

In [ ]:
BACKTEST_START = pd.Timestamp('20250101')
BACKTEST_END = pd.Timestamp('20251230')
INITIAL_CAPITAL = 1000000.0
TOP_N = 100
WEIGHT_CAP_MULTIPLIER = 1.5
BUY_RATE = 0.00031
SELL_RATE = 0.00081
RISK_WINDOW = 200
RISK_LAMBDA = 0.1

market_data = panel[[
    'trade_date', 'ts_code', 'open_hfq', 'close_hfq', 'ST', 'suspend'
]].copy().sort_values(['ts_code', 'trade_date'])
market_data['previous_close'] = (
    market_data.groupby('ts_code', sort=False)['close_hfq']
    .transform(lambda s: s.ffill().shift(1))
)
calendar = pd.DatetimeIndex(sorted(market_data['trade_date'].unique()))
previous_date = dict(zip(calendar[1:], calendar[:-1]))
forecast_by_date = {
    date: group.set_index('ts_code')['return_t2_pre']
    for date, group in stock_return.groupby('trade_date', sort=False)
}

# Keep the full history; today's return is excluded at each opening.
risk_returns = panel.pivot(index='trade_date', columns='ts_code', values='ret')
risk_returns = risk_returns.reindex(calendar)


## 风险惩罚优化与估值

使用 CVXPY 建模并调用 CLARABEL 求解器。变量为目标权重、买入权重和卖出权重，预算同时覆盖持仓和费用。优化失败直接报错，不设置替代策略。

目标为预测收益减买卖费用，再减 `0.1 * w.T @ covariance @ w`。协方差使用截至昨日最近 200 个市场交易日的 `panel.ret` 样本估计（`ddof=1`），不年化、不填缺失。先筛历史完整股票再取前 100 名，冻结持仓不参与风险项。首次建仓也进行风险优化，目标函数不扣交易费用惩罚；买入费用仍计入预算并实际扣除。后续调仓的目标包含交易费用惩罚。已有可交易旧持仓若历史不足则报错，不默默删除或虚构收益。

In [ ]:
def optimize_weights(mu, old_weights, covariance, initial_entry=False):
    n = len(mu)
    max_weight = WEIGHT_CAP_MULTIPLIER / n
    weights = cp.Variable(n)
    change = weights - old_weights
    cost = (
        BUY_RATE * cp.sum(cp.pos(change))
        + SELL_RATE * cp.sum(cp.pos(-change))
    )
    risk = cp.quad_form(weights, cp.psd_wrap(covariance))
    cost_penalty = 0.0 if initial_entry else cost
    problem = cp.Problem(
        cp.Maximize(mu @ weights - cost_penalty - RISK_LAMBDA * risk),
        [weights >= 0,
         weights <= max_weight,
         cp.sum(weights) + cost <= 1]
    )
    problem.solve(
        solver=cp.CLARABEL,
        tol_gap_abs=1e-8, tol_gap_rel=1e-8, tol_feas=1e-9, max_iter=500
    )
    if problem.status != cp.OPTIMAL:
        raise RuntimeError(f'Optimization status: {problem.status}')
    return np.maximum(weights.value, 0.0)


def valuation_prices(quotes, codes):
    return {
        code: float(quotes.loc[code, 'previous_close']
                    if quotes.loc[code, 'suspend']
                    else quotes.loc[code, 'open_hfq'])
        for code in codes
    }

## 单日交易

先执行 ST 或预测缺失强制退出并扣费，再更新候选池和旧权重。目标持仓数量按 `floor(weight * capital_liquid / open_hfq)` 向下取整，再卖出后买入，成本仅按实际差额交易计算。未用于交易的资金自然保留在 `cash`，不另行重复累加截尾差额。浮点或求解误差造成预算微小超出时，同比缩减买入数量后再次向下取整。数量为 0 的目标持仓正常卖出退出，冻结持仓数量保持不变。这里取整的是后复权合成数量 `quantity_hfq`，不是真实未复权股数或 100 股整手。

首次建仓先剔除 ST、停牌、预测缺失及最近 200 日收益历史不完整的股票，再取预测前 100 名。使用预测收益减风险惩罚优化权重，目标函数不扣交易费用惩罚，实际买入费用仍计入预算。单股上限为 `1.5 / n`，允许留现金，实际持仓数量由优化结果决定，不保证为 100。若无合格候选则继续持有现金，直到首次实际建仓。

In [ ]:
def execute_open(portfolio, cash, quotes, predictions, return_history, liquidate=False, initial_entry=False):
    quantities = portfolio.set_index('ts_code')['quantity_hfq'].to_dict()
    trades = []

    def trade(code, change, reason):
        nonlocal cash
        if change == 0:
            return
        price = float(quotes.loc[code, 'open_hfq'])
        amount = abs(change) * price
        rate = BUY_RATE if change > 0 else SELL_RATE
        fee = amount * rate
        cash += (-amount if change > 0 else amount) - fee
        quantities[code] = quantities.get(code, 0.0) + change
        if quantities[code] <= 0:
            quantities.pop(code, None)
        trades.append({
            'ts_code': code, 'side': 'buy' if change > 0 else 'sell',
            'quantity_hfq': abs(change), 'price_hfq': price,
            'amount': amount, 'fee': fee, 'reason': reason
        })

    # Mandatory sales happen before optimization; their fees are paid once.
    for code in list(quantities):
        if quotes.loc[code, 'suspend']:
            continue
        if liquidate:
            trade(code, -quantities[code], 'end_liquidation')
        elif quotes.loc[code, 'ST']:
            trade(code, -quantities[code], 'ST_exit')
        elif pd.isna(predictions.get(code, np.nan)):
            trade(code, -quantities[code], 'missing_prediction_exit')

    liquid_codes = [code for code in quantities if not quotes.loc[code, 'suspend']]
    capital_liquid = cash + sum(
        quantities[code] * quotes.loc[code, 'open_hfq'] for code in liquid_codes
    )
    candidate_count = 0
    if not liquidate and capital_liquid > 0:
        history_complete = return_history.notna().sum().eq(RISK_WINDOW)
        if len(return_history) != RISK_WINDOW:
            history_complete[:] = False
        if liquid_codes and not history_complete.reindex(liquid_codes, fill_value=False).all():
            raise RuntimeError('An existing tradable holding lacks 200 daily returns')
        eligible = quotes[['ST', 'suspend']].join(predictions.rename('prediction'))
        eligible = eligible.loc[
            eligible['ST'].eq(False) & eligible['suspend'].eq(False)
            & eligible['prediction'].notna()
            & eligible.index.isin(history_complete.index[history_complete])
        ]
        top_codes = (
            eligible.reset_index().sort_values(
                ['prediction', 'ts_code'], ascending=[False, True]
            ).head(TOP_N)['ts_code'].tolist()
        )
        candidates = sorted(set(top_codes) | set(liquid_codes))
        candidate_count = len(candidates)
        if candidates:
            prices = quotes.loc[candidates, 'open_hfq'].to_numpy()
            old_units = np.array([quantities.get(code, 0.0) for code in candidates])
            old_weights = old_units * prices / capital_liquid
            mu = predictions.reindex(candidates).to_numpy()
            # All stocks use the same 200 market dates, without filling.
            observations = return_history[candidates].to_numpy()
            centered = observations - observations.mean(axis=0)
            covariance = np.einsum('ti,tj->ij', centered, centered, optimize=False) / (RISK_WINDOW - 1)
            weights = optimize_weights(mu, old_weights, covariance, initial_entry=initial_entry)
            # Floor target synthetic quantities; unspent capital stays in cash.
            desired_units = np.floor(weights * capital_liquid / prices)
            changes = desired_units - old_units

            # Release cash from reductions before placing purchase orders.
            for code, change in zip(candidates, changes):
                if change < 0:
                    trade(code, float(change), 'rebalance')
            buy_total = float(np.maximum(changes, 0) @ prices)
            # Correct only possible solver/rounding overspend; never borrow.
            buy_scale = min(1.0, max(cash, 0.0) / (buy_total * (1 + BUY_RATE))) if buy_total > 0 else 1.0
            for code, change in zip(candidates, changes):
                if change > 0:
                    buy_units = int(np.floor(change * buy_scale))
                    trade(code, buy_units, 'initial_entry' if initial_entry else 'rebalance')

    if cash < -1e-6:
        raise RuntimeError(f'Negative cash after trading: {cash}')
    if abs(cash) < 1e-8:
        cash = 0.0
    updated = pd.DataFrame(
        [(code, quantity) for code, quantity in quantities.items() if quantity > 0],
        columns=['ts_code', 'quantity_hfq']
    )
    return updated, cash, trades, capital_liquid, candidate_count

## 逐日账户更新

生成每日账户、实际持仓和交易记录。逐日核对交易后资产等于交易前资产减费用；不重新运行旧等权策略。

In [ ]:
cash = INITIAL_CAPITAL
portfolio = pd.DataFrame(columns=['ts_code', 'quantity_hfq'])
previous_equity = INITIAL_CAPITAL
daily_records = []
holding_records = []
trade_records = []

for date, day in market_data.loc[
    market_data['trade_date'].between(BACKTEST_START, BACKTEST_END)
].groupby('trade_date', sort=True):
    quotes = day.set_index('ts_code')
    signal_date = previous_date.get(date, pd.NaT)
    predictions = forecast_by_date.get(signal_date, pd.Series(dtype=float))
    return_history = risk_returns.loc[risk_returns.index < date].tail(RISK_WINDOW)
    marks_before = valuation_prices(quotes, portfolio['ts_code'])
    equity_before = cash + sum(
        row.quantity_hfq * marks_before[row.ts_code]
        for row in portfolio.itertuples(index=False)
    )
    try:
        portfolio, cash, trades, capital_liquid, candidate_count = execute_open(
            portfolio, cash, quotes, predictions, return_history,
            liquidate=(date == BACKTEST_END),
            initial_entry=(len(trade_records) == 0)
        )
    except RuntimeError as error:
        raise RuntimeError(f'{date.date()}: {error}') from error

    marks = valuation_prices(quotes, portfolio['ts_code'])
    stock_value = 0.0
    frozen_value = 0.0
    for row in portfolio.itertuples(index=False):
        value = row.quantity_hfq * marks[row.ts_code]
        stock_value += value
        if quotes.loc[row.ts_code, 'suspend']:
            frozen_value += value
    total_equity = cash + stock_value
    total_fees = sum(trade['fee'] for trade in trades)
    if not np.isclose(total_equity, equity_before - total_fees, rtol=1e-10, atol=1e-5):
        raise RuntimeError(f'{date.date()}: account conservation failed')

    daily_records.append({
        'trade_date': date, 'signal_date': signal_date,
        'cash': cash, 'capital_liquid_before_optimization': capital_liquid,
        'liquid_stock_value': stock_value - frozen_value,
        'frozen_value': frozen_value, 'stock_value': stock_value,
        'total_equity': total_equity, 'fees': total_fees,
        'equity_change': total_equity - previous_equity,
        'daily_return': total_equity / previous_equity - 1,
        'nav': total_equity / INITIAL_CAPITAL,
        'n_stocks': len(portfolio), 'n_candidates': candidate_count,
        'gross_turnover': sum(trade['amount'] for trade in trades) / equity_before,
        'end_liquidation': date == BACKTEST_END
    })
    for row in portfolio.itertuples(index=False):
        value = row.quantity_hfq * marks[row.ts_code]
        holding_records.append({
            'trade_date': date, 'ts_code': row.ts_code,
            'quantity_hfq': row.quantity_hfq, 'valuation_price': marks[row.ts_code],
            'market_value': value, 'weight': value / total_equity,
            'frozen': bool(quotes.loc[row.ts_code, 'suspend'])
        })
    for trade_row in trades:
        trade_records.append({'trade_date': date, 'signal_date': signal_date, **trade_row})
    previous_equity = total_equity

backtest_daily = pd.DataFrame(daily_records)
backtest_holdings = pd.DataFrame(holding_records, columns=[
    'trade_date', 'ts_code', 'quantity_hfq', 'valuation_price',
    'market_value', 'weight', 'frozen'
])
backtest_trades = pd.DataFrame(trade_records, columns=[
    'trade_date', 'signal_date', 'ts_code', 'side', 'quantity_hfq',
    'price_hfq', 'amount', 'fee', 'reason'
])

## 预览账户与交易记录

依次查看账户表头尾、实际持仓和交易明细。现金包含取整后未使用的资金，持仓权重以交易后全账户资产为分母。


In [ ]:
backtest_daily.head()

In [ ]:
backtest_daily.tail()

In [ ]:
backtest_holdings.head()

In [ ]:
backtest_trades.head()

## 核心回测指标与同期指数

`backtest_summary` 仅保留 10 项核心指标：累计收益、年化收益、年化波动率、夏普比率、最大回撤、最差单日收益、累计费用、日均买卖合计换手、平均现金比例、期末冻结市值。定义见下方 `performance_metrics` 函数和汇总代码。

策略与 `index.parquet` 中指数使用相同起止日期及开盘到开盘口径。策略计入首次建仓与末日清仓费用；指数为不计费用的价格指数参考，不包含分红。年化按 252 个交易日、首尾开盘间隔数计算，现金与无风险收益率均设为 0。回撤只反映开盘采样。

### 对齐同期指数

按账户日期读取指数开盘点位，构造相同首尾日期的基准净值。指数不扣费用，首日收益率设为 0。


In [ ]:
index_data = pd.read_parquet(INDEX)
summary_daily = backtest_daily.sort_values('trade_date').set_index('trade_date')
benchmark_open = index_data.set_index('trade_date')['open'].reindex(summary_daily.index)
if benchmark_open.isna().any():
    raise ValueError('Index opening prices do not cover all backtest dates')
benchmark_nav = benchmark_open / benchmark_open.iloc[0]
benchmark_daily = pd.DataFrame({
    'nav': benchmark_nav,
    'daily_return': benchmark_nav.pct_change(fill_method=None).fillna(0)
})

### 收益风险指标

统一计算策略和指数的累计收益、年化收益、年化波动、夏普、最大回撤与最差单日收益。年化使用 252 个交易日，回撤包含初始净值 1。


In [ ]:
def performance_metrics(daily):
    nav = daily['nav']
    returns = daily['daily_return']
    intervals = len(daily) - 1
    annual_return = nav.iloc[-1] ** (252 / intervals) - 1 if intervals > 0 else np.nan
    daily_std = returns.std(ddof=1)
    sharpe = returns.mean() / daily_std * np.sqrt(252) if daily_std > 0 else np.nan
    # Initial capital is included so first-day fees count as drawdown.
    values = np.r_[1.0, nav.to_numpy()]
    drawdown = values / np.maximum.accumulate(values) - 1
    return {
        'total_return_pct': (nav.iloc[-1] - 1) * 100,
        'annual_return_pct': annual_return * 100,
        'annual_volatility_pct': daily_std * np.sqrt(252) * 100,
        'sharpe_rf0': sharpe,
        'max_drawdown_pct': drawdown.min() * 100,
        'worst_day_pct': returns.min() * 100
    }

### 汇总策略与指数

追加策略费用、平均买卖合计换手、现金比例和期末冻结市值，生成 `backtest_summary`；另保留两条净值序列作为 `backtest_comparison`。


In [ ]:
strategy_metrics = performance_metrics(summary_daily)
index_metrics = performance_metrics(benchmark_daily)
strategy_metrics.update({
    'total_fees': backtest_trades['fee'].sum(),
    'avg_gross_turnover_pct': summary_daily['gross_turnover'].mean() * 100,
    'avg_cash_weight_pct': (summary_daily['cash'] / summary_daily['total_equity']).mean() * 100,
    'final_frozen_value': summary_daily['frozen_value'].iloc[-1]
})
backtest_summary = pd.DataFrame({
    'strategy': pd.Series(strategy_metrics),
    'index': pd.Series(index_metrics)
})
backtest_summary.index.name = 'metric'

backtest_comparison = pd.DataFrame({
    'strategy_nav': summary_daily['nav'],
    'index_nav': benchmark_daily['nav']
})

### 展示汇总表

格式化展示核心指标。收益字段 `_pct` 已乘以 100，费用和冻结市值单位为元；结果保留在内存中，不自动保存文件。


In [ ]:
backtest_summary.style.format(
    lambda value: f'{value:,.4f}' if isinstance(value, (float, np.floating)) else str(value),
    na_rep='—'
)

## 报告图表：净值、回撤与交易行为

生成净值与回撤、月度复合收益、持仓数量与现金换手三张图。回撤包含初始净值1，月收益由每日净收益复合计算；期末清仓导致最后一天现金比例和卖出换手上升。图片保存到 `src/figures/`。


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

FIGURE_DIR = PROJECT_ROOT / 'src/figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10})

report_account = backtest_daily.sort_values('trade_date').set_index('trade_date')
report_nav = backtest_comparison[['strategy_nav', 'index_nav']].copy()
report_drawdown = report_nav.copy()
for col in report_nav:
    values = report_nav[col].to_numpy()
    report_drawdown[col] = values / np.maximum.accumulate(np.r_[1., values])[1:] - 1

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True,
                         gridspec_kw={'height_ratios': [2, 1]}, layout='constrained')
for col, name, color in [('strategy_nav', 'Strategy (after fees)', '#2563eb'),
                         ('index_nav', 'SSE A-share index', '#f59e0b')]:
    axes[0].plot(report_nav.index, report_nav[col], label=name, color=color)
    axes[1].plot(report_drawdown.index, report_drawdown[col], color=color)
axes[0].set(title='2025 performance: opening-to-opening', ylabel='NAV')
axes[0].legend(frameon=False)
axes[1].set(ylabel='Drawdown')
axes[1].yaxis.set_major_formatter(PercentFormatter(1))
for ax in axes:
    ax.grid(alpha=.2)
fig.savefig(FIGURE_DIR / 'backtest_nav_drawdown.png', dpi=200)
plt.show()

report_monthly = pd.DataFrame({
    'Strategy': (1 + report_account['daily_return']).resample('ME').prod() - 1,
    'Index': (1 + benchmark_daily['daily_return']).resample('ME').prod() - 1
})
fig, ax = plt.subplots(figsize=(12, 5), layout='constrained')
x = np.arange(len(report_monthly))
ax.bar(x - .2, report_monthly['Strategy'], width=.4, label='Strategy', color='#2563eb')
ax.bar(x + .2, report_monthly['Index'], width=.4, label='Index', color='#f59e0b')
ax.set_xticks(x, report_monthly.index.strftime('%Y-%m'), rotation=45, ha='right')
ax.axhline(0, color='#475569', linewidth=.8)
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set(title='Monthly compounded returns (through final liquidation)', ylabel='Monthly return')
ax.legend(frameon=False)
ax.grid(axis='y', alpha=.2)
fig.savefig(FIGURE_DIR / 'backtest_monthly_returns.png', dpi=200)
plt.show()

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True, layout='constrained')
axes[0].plot(report_account.index, report_account['n_stocks'], label='Actual holdings', color='#2563eb')
axes[0].plot(report_account.index, report_account['n_candidates'], label='Optimization candidates',
             color='#94a3b8', linestyle='--')
axes[0].set(ylabel='Stocks', title='Holdings, cash and trading activity')
axes[0].legend(frameon=False)
axes[1].plot(report_account.index, report_account['cash'] / report_account['total_equity'], color='#16a34a')
axes[1].set_ylabel('Cash weight')
axes[1].yaxis.set_major_formatter(PercentFormatter(1))
axes[2].bar(report_account.index, report_account['gross_turnover'], color='#64748b', width=1.5)
axes[2].set_ylabel('Buy + sell turnover')
axes[2].yaxis.set_major_formatter(PercentFormatter(1))
for ax in axes:
    ax.grid(alpha=.2)
fig.savefig(FIGURE_DIR / 'backtest_positions_turnover.png', dpi=200)
plt.show()
